In [1]:
import pandas as pd
import geopandas as gpd
import json
from shapely.affinity import translate
from helpers import remove_points_and_lines, calculate_intersection

In [2]:
def clean_ba_zone_geo(ba_system_geo, ba_zone_geo):
    unclaimed_areas = (
        ba_system_geo.geometry
        .difference(ba_zone_geo.union_all())
        .reset_index()
    )
    
    unclaimed_county_geo = calculate_intersection(
        unclaimed_areas,
        county_geo,
        ['rb']
    )
    
    unclaimed_counties = list(unclaimed_county_geo.rb.unique())
    
    zone_county_intersects = calculate_intersection(
        ba_zone_geo,
        county_geo.loc[county_geo.rb.isin(unclaimed_counties)],
        ['zone', 'rb']
    )
    zone_county_intersects['area'] = zone_county_intersects.geometry.area
    zone_county_intersects = (
        zone_county_intersects.sort_values('area', ascending=False)
        .drop_duplicates(subset='rb', keep='first')
    )
    county_zone_map = dict(zip(zone_county_intersects['rb'], zone_county_intersects['zone']))
    
    ba_county_geo_unassigned = county_geo.loc[(
        county_geo.rb.isin(unclaimed_counties) & ~county_geo.rb.isin(county_zone_map.keys())
    )]
    zone_county_nearest = gpd.sjoin_nearest(ba_zone_geo, ba_county_geo_unassigned, how='right')
    county_zone_map = (
        county_zone_map | dict(zip(zone_county_nearest['rb'], zone_county_nearest['zone']))
    )
    
    # Assign all unclaimed areas to their applicable zones
    unclaimed_county_geo['zone'] = unclaimed_county_geo['rb'].map(county_zone_map)
    unclaimed_area_geo = remove_points_and_lines(
        unclaimed_county_geo.dissolve('zone').reset_index(), ['zone']
    )
    
    # Append previously unclaimed areas to zone geo
    ba_zone_geo_all = remove_points_and_lines(
        pd.concat([ba_zone_geo, unclaimed_area_geo])
        .dissolve('zone')
        .reset_index(),
        ['zone']
    )
    ba_zone_geo_all = calculate_intersection(
        ba_zone_geo_all[['zone', 'geometry']],
        ba_system_geo,
        ['zone']
    )

    return ba_zone_geo_all

In [3]:
county_geo = gpd.read_file('data/shapefiles/US_COUNTY_2022')
bas = gpd.read_file('data/shapefiles/Balancing_Authorities').to_crs(county_geo.crs)
erst = gpd.read_file("data/shapefiles/Electric_Retail_Service_Territories").to_crs(county_geo.crs)

with open('config/rto_region_eia_code_map.json', 'r') as file:
    rto_region_id_eia_code_map = json.load(file)

rto_regions = gpd.read_file('data/shapefiles/RTO_Regions').to_crs(county_geo.crs)
rto_regions['EIAcode'] = rto_regions['Unique_ID'].map(rto_region_id_eia_code_map)
subbas = rto_regions.dropna(subset='EIAcode')
miso_new = subbas.loc[subbas.RTO_ISO == 'MISO'].dissolve(['EIAcode']).reset_index()
subbas = pd.concat([subbas.loc[subbas.RTO_ISO != 'MISO'], miso_new], ignore_index=True)

eia_930_ref = pd.read_excel('data/EIA930_Reference_Tables.xlsx', sheet_name='BA Subregions')
subba_code_name_map = dict(zip(eia_930_ref['BA Subregion Code'], eia_930_ref['BA Subregion Name']))

In [4]:
pnm_new = gpd.overlay(
    bas.loc[bas.EIAcode == 'PNM'],
    county_geo.loc[county_geo.STATE == 'Oregon'].dissolve(),
    how='difference'
)
bas = pd.concat([bas.loc[bas.EIAcode != 'PNM'], pnm_new], ignore_index=True)

In [5]:
wapa_geo = (
    erst.loc[(
        erst.CNTRL_AREA.isin([
            'WESTERN AREA POWER ADMINISTRATION - DESERT SOUTHWEST REGION',
            'WESTERN AREA POWER ADMINISTRATION - ROCKY MOUNTAIN REGION',
            'WESTERN AREA POWER ADMINISTRATION UGP WEST'
        ])
    )]
    .copy()
)
wapa_geo['geometry'] = wapa_geo.make_valid()
wapa_geo = wapa_geo[['CNTRL_AREA','geometry']].dissolve('CNTRL_AREA')

wapa_dsw_counties = [
    'p06079',
    'p06029',
    'p06071',
    'p06083',
    'p06111',
    'p06037',
    'p06059',
    'p06065',
    'p06073',
    'p06025',
    'p32003',
    'p32023',
    'p32017'
]
wapa_dsw_county_geo = (
    county_geo.loc[(
        county_geo.rb.isin(wapa_dsw_counties) | county_geo.STCODE.isin(['AZ', 'NM', 'UT'])
    )]
    [['geometry']]
    .dissolve()
)
wapa_dsw_geo = calculate_intersection(bas.loc[bas.EIAcode == 'WALC'], wapa_dsw_county_geo, [])

bas.loc[bas.EIAcode == 'WALC', 'geometry'] = wapa_dsw_geo.iloc[0]['geometry']

In [6]:
spp_region = rto_regions.loc[((rto_regions.RTO_ISO == 'SPP') & (rto_regions.Map_Type == 'Region'))].head(1)

spp_fill_in_features = gpd.read_file('data/shapefiles/spp_manually_drawn_zones')
spp_fill_in_features['geometry'] = (
    spp_fill_in_features['geometry']
    .apply(lambda geom: translate(geom, yoff=20000))
    .make_valid()
)
id_name_map = {
    2: 'WAUE',
    3: 'WAUE',
    99: 'SPS',
    98: 'CSWS',
    97: 'WFEC',
    95: 'OKGE',
    94: 'OKGE',
    93: 'CSWS',
    92: 'CSWS',
    91: 'CSWS',
    90: 'CSWS',
    89: 'GRDA',
    88: 'CSWS',
    87: 'EDE',
    85: 'MPS',
    81: 'SECI',
}
spp_fill_in_features['EIAcode'] = spp_fill_in_features['id'].map(id_name_map)
spp_fill_in_features = spp_fill_in_features.dissolve('EIAcode')

zones_no_overlap = {}
for zone, geo in spp_fill_in_features['geometry'].items():
    zones_no_overlap[zone] = geo.difference(spp_fill_in_features.drop(zone).union_all())

spp_fill_in_features = gpd.GeoDataFrame({
    'EIAcode': zones_no_overlap.keys(),
    'geometry': zones_no_overlap.values()
}, crs=spp_fill_in_features.crs)

In [7]:
waue_non_ne = calculate_intersection(
    spp_region,
    county_geo.loc[county_geo.STCODE.isin(['MT', 'ND', 'SD', 'MN', 'IA', 'WY'])].dissolve(),
    []
)
waue_ne = spp_fill_in_features.loc[spp_fill_in_features.EIAcode == 'WAUE'].copy()
waue_new = (
    pd.concat([waue_non_ne, waue_ne])
    .dissolve()
)

subbas.loc[subbas.EIAcode == 'WAUE', 'geometry'] = waue_new.iloc[0]['geometry']

In [8]:
ne_counties = county_geo.loc[county_geo.STATE == 'Nebraska'].copy()
ne_counties['county_area'] = ne_counties['geometry'].area
nppd_new = gpd.overlay(
    ne_counties,
    subbas.loc[subbas.EIAcode.isin(['WAUE', 'OPPD', 'LES'])].dissolve(),
    how='difference'
)
nppd_new['proportion_of_county_area'] = nppd_new.geometry.area / nppd_new['county_area']
nppd_new = (
    nppd_new.loc[nppd_new.proportion_of_county_area.round(1) > 0]
    .dissolve()
)
subbas.loc[subbas.EIAcode == 'NPPD', 'geometry'] = nppd_new.iloc[0]['geometry']

In [9]:
spp_fill_in_features_sub = (
    spp_fill_in_features.loc[~spp_fill_in_features.EIAcode.isin(['WAUE', 'SECI', 'MPS'])]
)
for eia_code in spp_fill_in_features_sub.EIAcode.tolist():
    subbas.loc[subbas.EIAcode == eia_code, 'geometry'] = (
        spp_fill_in_features_sub.loc[spp_fill_in_features_sub.EIAcode == eia_code].iloc[0]['geometry']
    )

In [10]:
ks_counties = county_geo.loc[county_geo.STATE == 'Kansas'].copy()
ks_counties['county_area'] = ks_counties['geometry'].area
wr_new = gpd.overlay(
    ks_counties,
    subbas.loc[(subbas.RTO_ISO == 'SPP') & (subbas.EIAcode != 'WR')].dissolve(),
    how='difference'
)
wr_new['proportion_of_county_area'] = wr_new.geometry.area / wr_new['county_area']
wr_new = (
    wr_new.loc[wr_new.proportion_of_county_area.round(1) > 0]
    .dissolve()
)
subbas.loc[subbas.EIAcode == 'WR', 'geometry'] = wr_new.iloc[0]['geometry']

In [11]:
subbas.loc[subbas.EIAcode == 'EDE', 'geometry'] = (
    gpd.overlay(
        subbas.loc[subbas.EIAcode == 'EDE'],
        subbas.loc[subbas.EIAcode.isin(['MPS', 'SPRM'])],
        how='difference'   
    )
    .iloc[0]
    ['geometry']
)

In [12]:
ba_system_geo = spp_region.copy()
ba_zone_geo = (
    subbas.loc[subbas.RTO_ISO == 'SPP']
    .copy()
    .rename(columns={'EIAcode': 'zone'})
)
spp_zones = clean_ba_zone_geo(ba_system_geo, ba_zone_geo)

In [13]:
subbas = (
    pd.concat([
        subbas.loc[subbas.RTO_ISO != 'SPP'],
        spp_zones.drop(columns='EIAcode').rename(columns={'zone': 'EIAcode'})
    ], ignore_index=True)
)

In [14]:
ba_system_geo = bas.loc[bas.EIAcode == 'PJM'].copy()

In [15]:
va_geo = (
    county_geo.loc[county_geo.STCODE == 'VA']
    .dissolve()
    ['geometry']
    .iloc[0]
)
va_no_dom_geo = gpd.GeoDataFrame(
    geometry=[va_geo.difference(erst.loc[erst.ID == '19876'].iloc[0]['geometry'])],
    crs=erst.crs
)
va_counties_no_dom_geo = calculate_intersection(
    va_no_dom_geo,
    county_geo.loc[county_geo.STCODE == 'VA'],
    ['rb']
)
pe_transmission_zone_geo = (
    va_counties_no_dom_geo.loc[(
        va_counties_no_dom_geo.NAME.isin([
            'Frederick',
            'Winchester',
            'Clarke',
            'Warren',
            'Page',
            'Rappahannock',
            'Madison',
            'Highland',
            'Greene'
        ])
    )]
    .dissolve()
)

In [16]:
with open("config/pjm_subregion_utility_map.json") as f:
    subregion_utilities = json.load(f)

subregion_utilities_reverse = {}
for k, v in subregion_utilities.items():
    if isinstance(v, str):
        subregion_utilities_reverse[v] = k
    else:
        for v_ in v:
            subregion_utilities_reverse[v_] = k

ba_utility_ids = sum(
    [[v] if isinstance(v, str) else v for v in subregion_utilities.values()],
    []
)

erst_ba = (
    erst.loc[erst.ID.isin([str(id) for id in ba_utility_ids])]
    [['NAME', 'ID', 'geometry']]
    .copy()
)

erst_ba['zone'] = erst_ba['ID'].astype(int).map(subregion_utilities_reverse)

erst_ba = pd.concat([
    erst_ba,
    pe_transmission_zone_geo.assign(zone='AP')
])

ba_zone_geo = erst_ba.dissolve('zone')[['geometry']]

# Modify Dominion service area
dom_geo = calculate_intersection(
    ba_system_geo,
    county_geo.loc[county_geo.STCODE.isin(['NC', 'VA'])].dissolve(),
    None
)
dom_geo.loc[0, 'geometry'] = (
    dom_geo.loc[0, 'geometry'].difference(ba_zone_geo.dissolve()['geometry'].iloc[0])
)
dom_geo = remove_points_and_lines(dom_geo)
ba_zone_geo = pd.concat([
    ba_zone_geo,
    dom_geo.assign(zone='DOM').set_index('zone')[['geometry']]
])
# Modify Potomac Electric service area
pep_geo = (
    pd.concat([
        county_geo.loc[(
            county_geo.rb.isin(['p24037', 'p24017', 'p24009', 'p24033'])
        )],
        erst.loc[erst.STATE == 'DC']
    ])
    .dissolve()
    [['geometry']]
)
pep_geo = remove_points_and_lines(pep_geo)
ba_zone_geo = pd.concat([
    ba_zone_geo.loc[ba_zone_geo.index != 'PEP'],
    pep_geo.assign(zone='PEP').set_index('zone')[['geometry']]
])

zones_no_overlap = {}
for zone, geo in ba_zone_geo['geometry'].items():
    zones_no_overlap[zone] = geo.difference(ba_zone_geo.drop(zone).union_all())

ba_zone_geo = gpd.GeoDataFrame({
    'zone': zones_no_overlap.keys(),
    'geometry': zones_no_overlap.values()
}, crs=ba_zone_geo.crs)

In [17]:
ba_system_geo['geometry'] = ba_system_geo.make_valid()
ba_zone_geo['geometry'] = ba_zone_geo.make_valid()
pjm_zones = clean_ba_zone_geo(ba_system_geo, ba_zone_geo)

In [25]:
aep_temp = gpd.overlay(
    pjm_zones.loc[pjm_zones.zone == 'AEP'],
    (
        pd.concat([
            erst.loc[erst.NAME.str.contains('WAYNETOWN|KINGSPORT|MIDWEST ENERGY COOPERATIVE')],
            county_geo.loc[county_geo.rb.isin(['p18015', 'p39037'])]
        ])
        .dissolve()
    ),
    how='difference'
)
aep_new = (
    pd.concat([
        aep_temp,
        erst.loc[erst.NAME.str.contains('APPALACHIAN POWER CO|INDIANA MICHIGAN|CITY OF NILES - \(MI\)')],
        county_geo.loc[county_geo.rb.isin(['p26027', 'p51195', 'p51720'])],
        calculate_intersection(
            erst.loc[erst.NAME.str.contains('INDIANA MICHIGAN|CITY OF NILES - \(MI\)|MIDWEST ENERGY COOP')].dissolve(),
            county_geo.loc[county_geo.rb.isin(['p26159', 'p26077', 'p26149', 'p26027', 'p51195', 'p51720', 'p26021'])].dissolve(),
            []
        ),
        county_geo.loc[(
            county_geo.rb.isin([
                'p51750',
                'p51690',
                'p51089',
                'p51121',
                'p51155',
                'p51027',
                'p51051',
                'p51195',
                'p51720',
                'p21195',
                'p51167',
                'p21193'
            ])
        )].dissolve()
    ])
    .dissolve()
)

pjm_zones.loc[pjm_zones.zone == 'AEP', 'geometry'] = (
    aep_new.iloc[0]['geometry']
)

pjm_zones.loc[pjm_zones.zone == 'ATSI', 'geometry'] = (
    gpd.overlay(
        pjm_zones.loc[pjm_zones.zone == 'ATSI'],
        county_geo.loc[county_geo.STCODE == 'MI'].dissolve(),
        how='difference'
    )
    .iloc[0]
    ['geometry']
)

pjm_zones.loc[pjm_zones.zone == 'DOM', 'geometry'] = (
    gpd.overlay(
        pjm_zones.loc[pjm_zones.zone == 'DOM'],
        county_geo.loc[county_geo.rb.isin(['p37145', 'p37077', 'p37181', 'p37185', 'p51195', 'p51720'])].dissolve(),
        how='difference'
    )
    .iloc[0]
    ['geometry']
)


pjm_zones.loc[pjm_zones.zone == 'DAY', 'geometry'] = (
    pd.concat([
        gpd.overlay(
            pjm_zones.loc[pjm_zones.zone == 'DAY'],
            county_geo.loc[county_geo.rb.isin(['p18177'])].dissolve(),
            how='difference'
        ),
        county_geo.loc[county_geo.rb == 'p39037']
    ])
    .dissolve()
    .iloc[0]
    ['geometry']
)

for eia_code in pjm_zones['zone'].tolist():
    if eia_code not in ['CE', 'EKPC']:
        subbas.loc[subbas.EIAcode == eia_code, 'geometry'] = (
            pjm_zones.loc[pjm_zones.zone == eia_code].iloc[0]['geometry']
        )

In [26]:
pjm_decomp = subbas.loc[subbas.EIAcode.isin(pjm_zones.zone)].explode()
pjm_decomp['area'] = pjm_decomp.geometry.area
pjm_decomp['zone_area'] = pjm_decomp.groupby('EIAcode')['area'].transform('sum')
pjm_decomp['prop_area'] = pjm_decomp['area'] / pjm_decomp['zone_area']
pjm_recomp = (
    pjm_decomp.loc[pjm_decomp.prop_area.round(3) > 0]
    .dissolve('EIAcode')
    .reset_index()
)
pjm_recomp = pd.concat([
    pjm_recomp.loc[pjm_recomp.EIAcode != 'EKPC'],
    subbas.loc[subbas.EIAcode == 'EKPC']
])

for eia_code in pjm_zones['zone'].tolist():
    subbas.loc[subbas.EIAcode == eia_code, 'geometry'] = (
        pjm_recomp.loc[pjm_recomp.EIAcode == eia_code].iloc[0]['geometry']
    )

In [27]:
subbas['EIAname'] = subbas['EIAcode'].map(subba_code_name_map)

In [28]:
eia_930_ref = pd.read_excel('data/EIA930_Reference_Tables.xlsx', sheet_name='BAs')
inactive_bas = eia_930_ref.loc[eia_930_ref['Active BA'] == 'No']['BA Code'].tolist()
inactive_bas.remove('WAUE')
gen_only_bas = eia_930_ref.loc[eia_930_ref['Generation Only BA'] == 'Yes']['BA Code'].tolist()
non_us_bas = eia_930_ref.loc[eia_930_ref['U.S. BA'] == 'No']['BA Code'].tolist()
parent_bas = ['CISO', 'ERCO', 'ISNE', 'MISO', 'NYIS', 'PJM', 'SWPP']
remove_bas = list(set(inactive_bas + gen_only_bas + non_us_bas + parent_bas + ['OVEC', 'SEC']))

eia_930_regions = (
    pd.concat([bas.loc[~bas.EIAcode.isin(remove_bas)], subbas], ignore_index=True)
    [['EIAcode', 'EIAname', 'geometry']]
)

In [29]:
eia_930_regions['geometry'] = eia_930_regions.make_valid()

In [37]:
eia_930_regions.to_file('data/shapefiles/bas_and_subbas')